In [38]:
ncpu=1  ## default set to be 1, meaning that one node has 1 core
mem= 8   ## memory 2 GiB
jobfs=200
walltime="22:00:00"  ## running time: 22 hours
path="/scratch/tq78/xl6985/tmp/NCI/"  ## need to change, from SSH 'cd /scratch' 'ls' 'cd vh89' 'cd username'

## if there is no own username, make a new one by 'mkdir jeffx9' 
## this is the final dist where the script should be, make sure to to consistence

node="normalbw"
project="tq78"  ## should be default
email="xian.li@anu.edu.au"

your information
_____________________________________________________________________________________

In [39]:
def batch_generator(i,ncpu,mem,jobfs,walltime,path,node,project,email):
    f = open("batch"+"_"+str(i), "w")
    f.write("#!/bin/bash\n")
    f.write("#PBS -l ncpus="+str(ncpu)+"\n")
    f.write("\n")
    f.write("#PBS -l mem="+str(mem)+"GB"+"\n")
    f.write("#PBS -l jobfs="+str(jobfs)+"GB"+"\n")
    f.write("#PBS -q "+node+"\n")
    f.write("#PBS -P "+project+"\n")
    f.write("#PBS -l walltime="+walltime+"\n")
    f.write("#PBS -l storage=gdata/dk92+gdata/"+project+"+scratch/"+project+"\n")
    #PBS -l storage=gdata/dk92+gdata/a00+scratch/a00
    f.write("#PBS -M "+email+"\n")

    #f.write("#PBS -l storage=gdata/a00+scratch/a00")
    f.write("#PBS -l wd"+"\n")
    #f.write("module use /g/data/dk92/apps/Modules/modulefiles"+"\n") ## may need to change
    f.write("module purge"+"\n")
    f.write("module load intel-compiler/2019.3.199"+"\n")
    f.write("module load intel-ipp/2019.3.199"+"\n") 
    f.write("module load intel-mkl/2019.3.199"+"\n")    
    f.write("module load R/3.6.1"+"\n")  
    #f.write("module load NCI-data-analysis/2024.05"+"\n") ## may need to change
    f.write("Rscript < "+path+"main_"+str(i)+".R"+" > $PBS_JOBID.log"+"\n")
    f.write("Rscript "+path+"main_"+str(i)+".R"+"\n")
    ##f.write("julia -p"+path+"main_"+str(i)+".jl"+" > $PBS_NCPUS.log"+"\n") 
    ##f.write("mpirun julia "+path+"main_"+str(i)+".jl"+" >& output.log"+"\n")
    ##f.write("mpirun julia "+"main_"+str(i)+".jl"+" >& output.log"+"\n")
    ##may need to change, previous one is for Python
    f.close()

In [40]:
if mem/(256/28)>1:
    ncpu=int(mem/(256/28))#the optimal cpu

In [41]:
sim_number=200 ## change to 200
step=1

In [42]:
for i in range(0,int(sim_number/step)):
    batch_generator(i,ncpu,mem,jobfs,walltime,path,node,project,email)

all batch files have been generated
_____________________________________________________________________________________

In [37]:
f = open("execute.sh", "w")
for i in range(0,int(sim_number/step)):
    f.write("qsub "+path+ "batch_"+str(i)+'\n')
f.close()

the execution file has been generated
_____________________________________________________________________________________

In [24]:
from shutil import copyfile ##the copy of main_0.py simulation file
for i in range(1,int(sim_number/step)):
    copyfile("main_0.R", "main_"+str(i)+".R") ##make sure you have your main_0.py in the folder

copy 'main_0.py', the first row to be 'rep_ind_current=0', treat as set.seed

In [25]:
for i in range(0,int(sim_number/step)): ### help to set seed the file of your simulation
    with open("main_"+str(i)+".R", 'r+') as f:
        content = f.read()
        f.seek(0, 0)
        f.write("rep_ind_current="+str(i*step) + '\n' + content)

the simulation files have been generated
_____________________________________________________________________________________

## Delete job

In [2]:
job_id_start=150303297
job_id_end=150303945
f = open("del.sh", "w") #execute this file in the gadi will delete the jobs from 16033220 to 16033319
for i in range(job_id_start,int(job_id_end)):
    f.write("qdel "+str(i)+ '.gadi-pbs\n')
f.close()

Final linux command: 'bash execute.sh'

Check memory: 'cat batch_100.o######'

'qstat' check staus